In [2]:
import os
import numpy as np
import librosa

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    GlobalAveragePooling1D,
    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.utils import to_categorical

# =====================================================
# DATASET
# =====================================================

DATASET = "/media/feliciano/Aux/AI_AFS_DATASET/classified"

CLASSES = [
    "prefeeding",
    "feeding",
    "postfeeding"
]

SR = 16000
DURATION = 2

TARGET_LENGTH = SR * DURATION

# =====================================================
# LOAD WAVS
# =====================================================

X = []
y = []

for label in CLASSES:

    folder = os.path.join(DATASET, label)

    files = [
        f for f in os.listdir(folder)
        if f.endswith(".wav")
    ]

    print(label, len(files))

    for wav in files:

        path = os.path.join(folder, wav)

        try:

            signal, _ = librosa.load(
                path,
                sr=SR,
                mono=True
            )

            if len(signal) < TARGET_LENGTH:

                signal = np.pad(
                    signal,
                    (
                        0,
                        TARGET_LENGTH - len(signal)
                    )
                )

            signal = signal[:TARGET_LENGTH]

            X.append(signal)
            y.append(label)

        except Exception as e:

            print("Error:", path, e)

# =====================================================
# NUMPY
# =====================================================

X = np.array(X, dtype=np.float32)
y = np.array(y)

print("X:", X.shape)

# =====================================================
# LABEL ENCODING
# =====================================================

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

y_cat = to_categorical(y_encoded)

# =====================================================
# TRAIN / TEST
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

# =====================================================
# CLASS WEIGHTS
# =====================================================

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_encoded),
    y=y_encoded
)

class_weights = {
    i: w
    for i, w in enumerate(weights)
}

print(class_weights)

# =====================================================
# MODEL
# =====================================================

model = Sequential([

    Conv1D(
        16,
        9,
        activation="relu",
        input_shape=(TARGET_LENGTH, 1)
    ),

    BatchNormalization(),

    MaxPooling1D(4),

    Conv1D(
        32,
        9,
        activation="relu"
    ),

    BatchNormalization(),

    MaxPooling1D(4),

    Conv1D(
        64,
        9,
        activation="relu"
    ),

    BatchNormalization(),

    MaxPooling1D(4),

    GlobalAveragePooling1D(),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        3,
        activation="softmax"
    )
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# =====================================================
# TRAIN
# =====================================================

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights
)

# =====================================================
# EVALUATION
# =====================================================

loss, acc = model.evaluate(
    X_test,
    y_test
)

print()
print("Test Accuracy:", acc)

# =====================================================
# SAVE
# =====================================================

model.save(
    "fish_feeding_raw_wav_cnn.keras"
)

print("Model saved.")

2026-07-29 15:54:17.603798: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785336857.721515 1596522 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785336857.821715 1596522 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1785336858.664862 1596522 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785336858.664919 1596522 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785336858.664925 1596522 computation_placer.cc:177] computation placer alr

prefeeding 2779
feeding 1320
postfeeding 2220
X: (6319, 32000)
{0: 1.5957070707070706, 1: 0.9487987987987988, 2: 0.7579465035384431}


/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1785336884.549237 1596522 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2707 MB memory:  -> device: 0, name: Quadro P2000, pci bus id: 0000:65:00.0, compute capability: 6.1


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 31992, 16)      │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 31992, 16)      │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 7998, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 7990, 32)       │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 7990, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 1997, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 1989, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1989, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 497, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

Epoch 1/30


I0000 00:00:1785336889.677532 1597131 service.cc:152] XLA service 0x7d1da4006bb0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1785336889.677561 1597131 service.cc:160]   StreamExecutor device (0): Quadro P2000, Compute Capability 6.1
2026-07-29 15:54:49.849312: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1785336890.249697 1597131 cuda_dnn.cc:529] Loaded cuDNN version 90201


  1/127 ━━━━━━━━━━━━━━━━━━━━ 22:29 11s/step - accuracy: 0.1875 - loss: 1.3295

I0000 00:00:1785336898.366145 1597131 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


127/127 ━━━━━━━━━━━━━━━━━━━━ 33s 173ms/step - accuracy: 0.4095 - loss: 1.0774 - val_accuracy: 0.3304 - val_loss: 1.1109
Epoch 2/30
127/127 ━━━━━━━━━━━━━━━━━━━━ 16s 127ms/step - accuracy: 0.4325 - loss: 1.0285 - val_accuracy: 0.2255 - val_loss: 1.1292
Epoch 3/30
127/127 ━━━━━━━━━━━━━━━━━━━━ 16s 127ms/step - accuracy: 0.4397 - loss: 1.0164 - val_accuracy: 0.3304 - val_loss: 1.1012
Epoch 4/30
127/127 ━━━━━━━━━━━━━━━━━━━━ 16s 128ms/step - accuracy: 0.4486 - loss: 1.0066 - val_accuracy: 0.2305 - val_loss: 1.2125
Epoch 5/30
127/127 ━━━━━━━━━━━━━━━━━━━━ 16s 128ms/step - accuracy: 0.4599 - loss: 0.9993 - val_accuracy: 0.3323 - val_loss: 1.1021
Epoch 6/30
127/127 ━━━━━━━━━━━━━━━━━━━━ 16s 127ms/step - accuracy: 0.4508 - loss: 1.0011 - val_accuracy: 0.4441 - val_loss: 3.5677
Epoch 7/30
127/127 ━━━━━━━━━━━━━━━━━━━━ 16s 128ms/step - accuracy: 0.4614 - loss: 0.9956 - val_accuracy: 0.3304 - val_loss: 16.3281
Epoch 8/30
127/127 ━━━━━━━━━━━━━━━━━━━━ 16s 128ms/step - accuracy: 0.4545 - loss: 0.9915 - va

In [3]:
import os
import librosa
import numpy as np
import pandas as pd

DATASET = "/media/feliciano/Aux/AI_AFS_DATASET/classified"

classes = [
    "prefeeding",
    "feeding",
    "postfeeding"
]

rows = []

for label in classes:

    folder = os.path.join(DATASET, label)

    files = [
        f for f in os.listdir(folder)
        if f.endswith(".wav")
    ]

    print(label, len(files))

    for file in files:

        path = os.path.join(folder, file)

        try:

            y, sr = librosa.load(
                path,
                sr=16000,
                mono=True
            )

            y = (y - np.mean(y)) / (
                np.std(y) + 1e-8
            )

            rms = np.mean(
                librosa.feature.rms(y=y)
            )

            zcr = np.mean(
                librosa.feature.zero_crossing_rate(y)
            )

            centroid = np.mean(
                librosa.feature.spectral_centroid(
                    y=y,
                    sr=sr
                )
            )

            bandwidth = np.mean(
                librosa.feature.spectral_bandwidth(
                    y=y,
                    sr=sr
                )
            )

            rolloff = np.mean(
                librosa.feature.spectral_rolloff(
                    y=y,
                    sr=sr
                )
            )

            contrast = np.mean(
                librosa.feature.spectral_contrast(
                    y=y,
                    sr=sr
                )
            )

            mfcc = librosa.feature.mfcc(
                y=y,
                sr=sr,
                n_mfcc=13
            )

            mfcc_means = np.mean(
                mfcc,
                axis=1
            )

            row = {
                "file": file,
                "class": label,
                "rms": rms,
                "zcr": zcr,
                "centroid": centroid,
                "bandwidth": bandwidth,
                "rolloff": rolloff,
                "contrast": contrast
            }

            for i in range(13):

                row[f"mfcc_{i+1}"] = mfcc_means[i]

            rows.append(row)

        except Exception as e:

            print(path, e)

df = pd.DataFrame(rows)

df.to_csv(
    "fish_features.csv",
    index=False
)

print(df.shape)
print("Saved fish_features.csv")


prefeeding 2779
feeding 1320
postfeeding 2220
(6319, 21)
Saved fish_features.csv


In [4]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

# =====================================================
# LOAD FEATURES
# =====================================================

df = pd.read_csv("fish_features.csv")

X = df.drop(
    columns=["file", "class"]
)

y = df["class"]

# =====================================================
# SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# =====================================================
# MODEL
# =====================================================

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)

# =====================================================
# PREDICTIONS
# =====================================================

pred = model.predict(X_test)

# =====================================================
# RESULTS
# =====================================================

print("\nAccuracy:")
print(
    accuracy_score(y_test, pred)
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        pred
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        pred
    )
)

# =====================================================
# FEATURE IMPORTANCE
# =====================================================

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

print("\nTop 20 Features:")
print(
    importance.head(20)
)

# =====================================================
# SAVE MODEL
# =====================================================

joblib.dump(
    model,
    "fish_random_forest.pkl"
)

print(
    "\nModel saved: fish_random_forest.pkl"
)


Accuracy:
0.6257911392405063

Classification Report:
              precision    recall  f1-score   support

     feeding       0.72      0.32      0.45       264
 postfeeding       0.58      0.66      0.62       444
  prefeeding       0.65      0.74      0.69       556

    accuracy                           0.63      1264
   macro avg       0.65      0.58      0.58      1264
weighted avg       0.64      0.63      0.61      1264


Confusion Matrix:
[[ 85  83  96]
 [ 19 295 130]
 [ 14 131 411]]

Top 20 Features:
      feature  importance
10     mfcc_5    0.064293
15    mfcc_10    0.063272
18    mfcc_13    0.063234
14     mfcc_9    0.060473
11     mfcc_6    0.060218
13     mfcc_8    0.059323
16    mfcc_11    0.056831
17    mfcc_12    0.056579
12     mfcc_7    0.055580
5    contrast    0.053743
7      mfcc_2    0.053418
9      mfcc_4    0.049889
4     rolloff    0.045727
2    centroid    0.045149
1         zcr    0.044826
3   bandwidth    0.043923
8      mfcc_3    0.043014
6      mfcc_1 